In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
%cd "/content/drive/MyDrive/nearest_neighbors/"

/content/drive/MyDrive/nearest_neighbors


In [ ]:
import os
from collections import Counter
from itertools import chain
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
torch.cuda.empty_cache()
import json
import torch.nn.functional as F
from nss_model.data_processing.dataloader import Dataloader
from nss_model.data_processing.dataset import Dataset
from nss_model.model.query_encoder import QueryEncoder
from nss_model.model.document_encoder import DocumentEncoder
from nss_model.model.train import Trainer

In [ ]:
BASE_DIR = os.path.abspath(".")

In [ ]:
dataset_path = os.path.join(BASE_DIR, "data/dataset/") + "aggregated_qd_dataset2.json"
dataloader_instance = Dataloader(batch_size=8, dataset_path=dataset_path, test_size=0.2)
train_dataloader = dataloader_instance.get_train_dataloader()

for batch in train_dataloader:
    print(len(batch))
    print("Keys in the batch:")
    for key in batch.keys():
        print(key)

    print("Shapes of tensors in the batch:")
    for key, tensor in batch.items():
         print(f"{key}: {tensor.shape}")
    break


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Dataset split into 32000 training and 8000 testing samples.
5
Keys in the batch:
query_input_ids
query_attention_mask
document_input_ids
document_attention_mask
document_numerical_features
Shapes of tensors in the batch:
query_input_ids: torch.Size([8, 128])
query_attention_mask: torch.Size([8, 128])
document_input_ids: torch.Size([8, 128])
document_attention_mask: torch.Size([8, 128])
document_numerical_features: torch.Size([8, 6])


In [ ]:
torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
query_encoder = QueryEncoder(bert_model_name='bert-base-uncased')
document_encoder = DocumentEncoder(
    bert_model_name='bert-base-uncased',
    numerical_dim=6,
    hidden_dim=128
)

trainer = Trainer(
    query_encoder=query_encoder,
    document_encoder=document_encoder,
    device=device,
    temperature=0.07,
    lr=2e-5,
    save_dir="checkpoints"
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [ ]:
num_epochs = 5
loss_per_epoch = trainer.train(train_dataloader, num_epochs=num_epochs)
loss_per_epoch = list(enumerate(loss_per_epoch, start=1))
best_epoch, best_loss = min(loss_per_epoch, key=lambda x: x[1])
query_path = f"checkpoints/epoch_{best_epoch}_query_encoder.pt"
doc_path = f"checkpoints/epoch_{best_epoch}_document_encoder.pt"
print(f"\n Best Epoch: {best_epoch} with Loss = {best_loss:.4f}")
print(f"Saved Weights:")
print(f" - Query Encoder: {query_path}")
print(f" - Document Encoder: {doc_path}")

Epoch: 1:   0%|          | 0/4000 [00:01<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
test_dataloader = dataloader_instance.get_test_dataloader()


trainer.load(
    "/content/drive/MyDrive/nearest_neighbors/checkpoints/epoch_5_query_encoder.pt",
    "/content/drive/MyDrive/nearest_neighbors/checkpoints/epoch_5_document_encoder.pt"
)


In [ ]:
print(logits.shape)